**Notas**

- Notebook basado en el curso de Andrew Ng, *Deep Learning Specialization* (DeepLearning.AI / Coursera).
- Modificado y editado con la ayuda de Claude.
- Revisado y editado por el autor final: Alvaro Riascos.
- Todos los errores son responsabilidad exclusiva del autor.

**Versión taller**

Completa el código en cada celda marcada con `# TU CÓDIGO EMPIEZA AQUÍ` y `# TU CÓDIGO TERMINA AQUÍ`. Los comentarios que están justo antes de cada bloque indican cuántas líneas se esperan y qué variables debes definir. Después de cada ejercicio, ejecuta la celda de prueba que sigue y compara con el **Resultado esperado**. No cambies los nombres de las funciones ni de sus argumentos: las pruebas automáticas dependen de ellos.

# Regresión logística con mentalidad de red neuronal

En este notebook construirás un clasificador de regresión logística para reconocer gatos. El notebook te guiará paso a paso para hacerlo con una *mentalidad de red neuronal*, y te ayudará a afinar tus intuiciones sobre el aprendizaje profundo (*deep learning*).

**Instrucciones:**
- No uses ciclos (for/while) en tu código, a menos que las instrucciones lo pidan explícitamente.
- Usa `np.dot(X,Y)` para calcular productos punto.

**Aprenderás a:**
- Construir la arquitectura general de un algoritmo de aprendizaje, incluyendo:
    - Inicializar los parámetros
    - Calcular la función de costo y su gradiente
    - Usar un algoritmo de optimización (descenso por gradiente)
- Reunir las tres funciones anteriores, en el orden correcto, en una función principal `model()`.

## Contenido
- [1 - Paquetes](#1)
- [2 - Descripción del problema](#2)
    - [Ejercicio 1](#ex-1)
    - [Ejercicio 2](#ex-2)
- [3 - Arquitectura general del algoritmo de aprendizaje](#3)
- [4 - Construcción de las partes del algoritmo](#4)
    - [4.1 - Funciones auxiliares](#4-1)
        - [Ejercicio 3 - sigmoid](#ex-3)
    - [4.2 - Inicialización de parámetros](#4-2)
        - [Ejercicio 4 - initialize_with_zeros](#ex-4)
    - [4.3 - Propagación hacia adelante y hacia atrás](#4-3)
        - [Ejercicio 5 - propagate](#ex-5)
    - [4.4 - Optimización](#4-4)
        - [Ejercicio 6 - optimize](#ex-6)
        - [Ejercicio 7 - predict](#ex-7)
- [5 - Integrar todas las funciones en un modelo](#5)
    - [Ejercicio 8 - model](#ex-8)
- [6 - Análisis adicional (opcional)](#6)
- [7 - Prueba con tu propia imagen (opcional)](#7)

<a name='1'></a>
## 1 - Paquetes ##

Primero, ejecuta la celda de abajo para importar todos los paquetes que necesitarás en este notebook.
- [numpy](https://numpy.org/doc/stable/) es el paquete fundamental para computación científica en Python.
- [h5py](http://www.h5py.org) es un paquete común para interactuar con datos almacenados en un archivo H5.
- [matplotlib](http://matplotlib.org) es una librería muy conocida para hacer gráficas en Python.
- [PIL](https://pillow.readthedocs.io/en/stable/) y [scipy](https://www.scipy.org/) se usan al final para probar el modelo con tu propia imagen.

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage
from lr_utils import load_dataset
from public_tests import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

<a name='2'></a>
## 2 - Descripción del problema ##

**Planteamiento del problema**: tienes un conjunto de datos ("data.h5") que contiene:
- un conjunto de entrenamiento de m_train imágenes etiquetadas como gato (y=1) o no-gato (y=0)
- un conjunto de prueba de m_test imágenes etiquetadas como gato o no-gato
- cada imagen tiene forma (num_px, num_px, 3), donde 3 corresponde a los 3 canales de color (RGB). Así, cada imagen es cuadrada (alto = num_px) y (ancho = num_px).

Construirás un algoritmo sencillo de reconocimiento de imágenes que clasifique correctamente las fotos como gato o no-gato.

Familiaricémonos con los datos. Cárgalos ejecutando el siguiente código.

In [ ]:
# Cargar los datos (gato/no-gato)
train_set_x_orig, train_set_y, test_set_x_orig, test_set_y, classes = load_dataset()

Agregamos "_orig" al final de los nombres de los conjuntos de imágenes (entrenamiento y prueba) porque vamos a preprocesarlos. Después del preprocesamiento tendremos train_set_x y test_set_x (las etiquetas train_set_y y test_set_y no necesitan preprocesamiento).

Cada fila de train_set_x_orig y test_set_x_orig es un arreglo que representa una imagen. Puedes visualizar un ejemplo ejecutando el siguiente código. Cambia el valor de `index` y vuelve a ejecutar para ver otras imágenes.

In [ ]:
# Ejemplo de una imagen
index = 25
plt.imshow(train_set_x_orig[index])
print ("y = " + str(train_set_y[:, index]) + ", es una imagen de tipo '" + classes[np.squeeze(train_set_y[:, index])].decode("utf-8") +  "'.")

Muchos errores de programación en *deep learning* provienen de dimensiones de matrices/vectores que no coinciden. Si mantienes claras las dimensiones de tus matrices y vectores, evitarás buena parte de esos errores.

<a name='ex-1'></a>
### Ejercicio 1
Encuentra los valores de:
- m_train (número de ejemplos de entrenamiento)
- m_test (número de ejemplos de prueba)
- num_px (= alto = ancho de una imagen de entrenamiento)

Recuerda que `train_set_x_orig` es un arreglo de numpy de forma (m_train, num_px, num_px, 3). Por ejemplo, puedes obtener `m_train` escribiendo `train_set_x_orig.shape[0]`.

In [ ]:
#(≈ 3 líneas de código)
# m_train =
# m_test =
# num_px =
# TU CÓDIGO EMPIEZA AQUÍ


# TU CÓDIGO TERMINA AQUÍ

print ("Número de ejemplos de entrenamiento: m_train = " + str(m_train))
print ("Número de ejemplos de prueba: m_test = " + str(m_test))
print ("Alto/Ancho de cada imagen: num_px = " + str(num_px))
print ("Cada imagen tiene tamaño: (" + str(num_px) + ", " + str(num_px) + ", 3)")
print ("Forma de train_set_x: " + str(train_set_x_orig.shape))
print ("Forma de train_set_y: " + str(train_set_y.shape))
print ("Forma de test_set_x: " + str(test_set_x_orig.shape))
print ("Forma de test_set_y: " + str(test_set_y.shape))

**Resultado esperado para m_train, m_test y num_px**:
<table style="width:15%">
  <tr>
    <td> m_train </td>
    <td> 209 </td>
  </tr>

  <tr>
    <td>m_test</td>
    <td> 50 </td>
  </tr>

  <tr>
    <td>num_px</td>
    <td> 64 </td>
  </tr>

</table>


Por conveniencia, ahora debes transformar las imágenes de forma (num_px, num_px, 3) en arreglos de numpy de forma (num_px $*$ num_px $*$ 3, 1). Después de esto, el conjunto de entrenamiento (y el de prueba) es un arreglo de numpy donde cada columna representa una imagen "aplanada". Debe haber m_train (respectivamente m_test) columnas.

<a name='ex-2'></a>
### Ejercicio 2
Cambia la forma de los conjuntos de entrenamiento y prueba para que las imágenes de tamaño (num_px, num_px, 3) queden aplanadas en vectores de forma (num\_px $*$ num\_px $*$ 3, 1).

Un truco para aplanar una matriz X de forma (a,b,c,d) en una matriz X_flatten de forma (b$*$c$*$d, a) es usar:
```python
X_flatten = X.reshape(X.shape[0], -1).T      # X.T es la transpuesta de X
```

In [ ]:
# Cambiar la forma de los ejemplos de entrenamiento y prueba
#(≈ 2 líneas de código)
# train_set_x_flatten = ...
# test_set_x_flatten = ...
# TU CÓDIGO EMPIEZA AQUÍ


# TU CÓDIGO TERMINA AQUÍ

# Verificar que los primeros 10 píxeles de la segunda imagen están en el lugar correcto
assert np.all(train_set_x_flatten[0:10, 1] == [196, 192, 190, 193, 186, 182, 188, 179, 174, 213]), "Solución incorrecta. Usa (X.shape[0], -1).T."
assert np.all(test_set_x_flatten[0:10, 1] == [115, 110, 111, 137, 129, 129, 155, 146, 145, 159]), "Solución incorrecta. Usa (X.shape[0], -1).T."

print ("Forma de train_set_x_flatten: " + str(train_set_x_flatten.shape))
print ("Forma de train_set_y: " + str(train_set_y.shape))
print ("Forma de test_set_x_flatten: " + str(test_set_x_flatten.shape))
print ("Forma de test_set_y: " + str(test_set_y.shape))

**Resultado esperado**:

<table style="width:35%">
  <tr>
    <td>Forma de train_set_x_flatten</td>
    <td> (12288, 209)</td>
  </tr>
  <tr>
    <td>Forma de train_set_y</td>
    <td>(1, 209)</td>
  </tr>
  <tr>
    <td>Forma de test_set_x_flatten</td>
    <td>(12288, 50)</td>
  </tr>
  <tr>
    <td>Forma de test_set_y</td>
    <td>(1, 50)</td>
  </tr>
</table>

Para representar imágenes a color, se deben especificar los canales rojo, verde y azul (RGB) de cada píxel; por eso el valor de un píxel es en realidad un vector de tres números entre 0 y 255.

Un paso de preprocesamiento común en *machine learning* es centrar y estandarizar los datos: restar a cada ejemplo la media de todo el arreglo y luego dividir por la desviación estándar de todo el arreglo. Pero para conjuntos de imágenes es más sencillo, más conveniente y funciona casi igual de bien dividir cada fila del conjunto por 255 (el valor máximo de un canal de píxel).

<!-- Durante el entrenamiento multiplicarás pesos y sumarás sesgos a las entradas para observar las activaciones de las neuronas. Luego harás retropropagación con los gradientes para entrenar el modelo. Es muy importante que cada variable tenga un rango similar para que los gradientes no exploten. !-->

Estandaricemos nuestros datos.

In [ ]:
train_set_x = train_set_x_flatten / 255.
test_set_x = test_set_x_flatten / 255.

<font color='blue'>


**Lo que debes recordar:**

Los pasos comunes para preprocesar un nuevo conjunto de datos son:
- Identificar las dimensiones y formas del problema (m_train, m_test, num_px, ...)
- Cambiar la forma de los datos para que cada ejemplo sea un vector de tamaño (num_px \* num_px \* 3, 1)
- "Estandarizar" los datos

<a name='3'></a>
## 3 - Arquitectura general del algoritmo de aprendizaje ##

Es hora de diseñar un algoritmo sencillo para distinguir imágenes de gatos de imágenes que no son de gatos.

Construirás una regresión logística con una mentalidad de red neuronal. La siguiente figura explica por qué **¡la regresión logística es en realidad una red neuronal muy sencilla!**

<img src="images/LogReg_kiank.png" style="width:650px;height:400px;">

**Expresión matemática del algoritmo**:

Para un ejemplo $x^{(i)}$:
$$z^{(i)} = w^T x^{(i)} + b \tag{1}$$
$$\hat{y}^{(i)} = a^{(i)} = sigmoid(z^{(i)})\tag{2}$$
$$ \mathcal{L}(a^{(i)}, y^{(i)}) =  - y^{(i)}  \log(a^{(i)}) - (1-y^{(i)} )  \log(1-a^{(i)})\tag{3}$$

El costo se calcula promediando sobre todos los ejemplos de entrenamiento:
$$ J = \frac{1}{m} \sum_{i=1}^m \mathcal{L}(a^{(i)}, y^{(i)})\tag{6}$$

**Pasos clave**:
En este ejercicio realizarás los siguientes pasos:
- Inicializar los parámetros del modelo
- Aprender los parámetros del modelo minimizando el costo
- Usar los parámetros aprendidos para hacer predicciones (sobre el conjunto de prueba)
- Analizar los resultados y concluir

<a name='4'></a>
## 4 - Construcción de las partes del algoritmo ##

Los pasos principales para construir una red neuronal son:
1. Definir la estructura del modelo (por ejemplo, el número de variables de entrada)
2. Inicializar los parámetros del modelo
3. Iterar:
    - Calcular la pérdida actual (propagación hacia adelante)
    - Calcular el gradiente actual (propagación hacia atrás)
    - Actualizar los parámetros (descenso por gradiente)

Usualmente se construyen los pasos 1-3 por separado y luego se integran en una sola función que llamamos `model()`.

<a name='4-1'></a>
### 4.1 - Funciones auxiliares

<a name='ex-3'></a>
### Ejercicio 3 - sigmoid
Usando tu código de "Python Basics", implementa `sigmoid()`. Como viste en la figura anterior, necesitas calcular $sigmoid(z) = \frac{1}{1 + e^{-z}}$ para $z = w^T x + b$ para hacer predicciones. Usa np.exp().

In [ ]:
# FUNCIÓN CALIFICADA: sigmoid

def sigmoid(z):
    """
    Calcula la sigmoide de z

    Argumentos:
    z -- un escalar o un arreglo de numpy de cualquier tamaño.

    Retorna:
    s -- sigmoid(z)
    """

    #(≈ 1 línea de código)
    # s = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    return s

In [ ]:
print ("sigmoid([0, 2]) = " + str(sigmoid(np.array([0,2]))))

sigmoid_test(sigmoid)

In [ ]:
x = np.array([0.5, 0, 2.0])
output = sigmoid(x)
print(output)

<a name='4-2'></a>
### 4.2 - Inicialización de parámetros

<a name='ex-4'></a>
### Ejercicio 4 - initialize_with_zeros
Implementa la inicialización de parámetros en la celda de abajo. Debes inicializar w como un vector de ceros. Si no sabes qué función de numpy usar, busca np.zeros() en la documentación de Numpy.

In [ ]:
# FUNCIÓN CALIFICADA: initialize_with_zeros

def initialize_with_zeros(dim):
    """
    Esta función crea un vector de ceros de forma (dim, 1) para w e inicializa b en 0.

    Argumento:
    dim -- tamaño del vector w que queremos (es decir, el número de parámetros)

    Retorna:
    w -- vector inicializado de forma (dim, 1)
    b -- escalar inicializado (corresponde al sesgo) de tipo float
    """

    # (≈ 2 líneas de código)
    # w = ...
    # b = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    return w, b

In [ ]:
dim = 2
w, b = initialize_with_zeros(dim)

assert type(b) == float
print ("w = " + str(w))
print ("b = " + str(b))

initialize_with_zeros_test_1(initialize_with_zeros)
initialize_with_zeros_test_2(initialize_with_zeros)


<a name='4-3'></a>
### 4.3 - Propagación hacia adelante y hacia atrás

Ahora que los parámetros están inicializados, puedes implementar los pasos de propagación "hacia adelante" (*forward*) y "hacia atrás" (*backward*) para aprender los parámetros.

<a name='ex-5'></a>
### Ejercicio 5 - propagate
Implementa una función `propagate()` que calcule la función de costo y su gradiente.

**Pistas**:

Propagación hacia adelante:
- Recibes X
- Calculas $A = \sigma(w^T X + b) = (a^{(1)}, a^{(2)}, ..., a^{(m-1)}, a^{(m)})$
- Calculas la función de costo: $J = -\frac{1}{m}\sum_{i=1}^{m}(y^{(i)}\log(a^{(i)})+(1-y^{(i)})\log(1-a^{(i)}))$

Estas son las dos fórmulas que usarás:

$$ \frac{\partial J}{\partial w} = \frac{1}{m}X(A-Y)^T\tag{7}$$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (a^{(i)}-y^{(i)})\tag{8}$$

In [ ]:
# FUNCIÓN CALIFICADA: propagate

def propagate(w, b, X, Y):
    """
    Implementa la función de costo y su gradiente para la propagación explicada arriba

    Argumentos:
    w -- pesos, un arreglo de numpy de tamaño (num_px * num_px * 3, 1)
    b -- sesgo, un escalar
    X -- datos de tamaño (num_px * num_px * 3, número de ejemplos)
    Y -- vector de "etiquetas" verdaderas (0 si no es gato, 1 si es gato) de tamaño (1, número de ejemplos)

    Retorna:
    grads -- diccionario con los gradientes de los pesos y del sesgo
            (dw -- gradiente de la pérdida con respecto a w, por lo tanto con la misma forma que w)
            (db -- gradiente de la pérdida con respecto a b, por lo tanto con la misma forma que b)
    cost -- costo de log-verosimilitud negativa para la regresión logística

    Consejos:
    - Escribe el código de la propagación paso a paso. np.log(), np.dot()
    """

    m = X.shape[1]

    # PROPAGACIÓN HACIA ADELANTE (DE X AL COSTO)
    #(≈ 2 líneas de código)
    # calcular la activación
    # A = ...
    # calcular el costo usando np.dot para la multiplicación.
    # Y no uses ciclos para la suma.
    # cost = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # PROPAGACIÓN HACIA ATRÁS (PARA ENCONTRAR EL GRADIENTE)
    #(≈ 2 líneas de código)
    # dw = ...
    # db = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ
    cost = np.squeeze(np.array(cost))


    grads = {"dw": dw,
             "db": db}

    return grads, cost

In [ ]:
w =  np.array([[1.], [2]])
b = 1.5
X = np.array([[1., -2., -1.], [3., 0.5, -3.2]])
Y = np.array([[1, 1, 0]])
grads, cost = propagate(w, b, X, Y)

assert type(grads["dw"]) == np.ndarray
assert grads["dw"].shape == (2, 1)
assert type(grads["db"]) == np.float64


print ("dw = " + str(grads["dw"]))
print ("db = " + str(grads["db"]))
print ("cost = " + str(cost))

propagate_test(propagate)

**Resultado esperado**

```
dw = [[ 0.25071532]
 [-0.06604096]]
db = -0.1250040450043965
cost = 0.15900537707692405
```

<a name='4-4'></a>
### 4.4 - Optimización
- Ya inicializaste los parámetros.
- También puedes calcular la función de costo y su gradiente.
- Ahora quieres actualizar los parámetros usando descenso por gradiente.

<a name='ex-6'></a>
### Ejercicio 6 - optimize
Escribe la función de optimización. El objetivo es aprender $w$ y $b$ minimizando la función de costo $J$. Para un parámetro $\theta$, la regla de actualización es $ \theta = \theta - \alpha \text{ } d\theta$, donde $\alpha$ es la tasa de aprendizaje (*learning rate*).

In [ ]:
# FUNCIÓN CALIFICADA: optimize

def optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False):
    """
    Esta función optimiza w y b ejecutando un algoritmo de descenso por gradiente

    Argumentos:
    w -- pesos, un arreglo de numpy de tamaño (num_px * num_px * 3, 1)
    b -- sesgo, un escalar
    X -- datos de forma (num_px * num_px * 3, número de ejemplos)
    Y -- vector de "etiquetas" verdaderas (0 si no es gato, 1 si es gato), de forma (1, número de ejemplos)
    num_iterations -- número de iteraciones del ciclo de optimización
    learning_rate -- tasa de aprendizaje de la regla de actualización del descenso por gradiente
    print_cost -- True para imprimir la pérdida cada 100 pasos

    Retorna:
    params -- diccionario con los pesos w y el sesgo b
    grads -- diccionario con los gradientes de los pesos y del sesgo con respecto a la función de costo
    costs -- lista de todos los costos calculados durante la optimización; se usa para graficar la curva de aprendizaje.

    Consejos:
    Básicamente debes escribir dos pasos e iterarlos:
        1) Calcular el costo y el gradiente para los parámetros actuales. Usa propagate().
        2) Actualizar los parámetros usando la regla del descenso por gradiente para w y b.
    """

    w = copy.deepcopy(w)
    b = copy.deepcopy(b)

    costs = []

    for i in range(num_iterations):
        # (≈ 1 línea de código)
        # Cálculo del costo y del gradiente
        # grads, cost = ...
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        # TU CÓDIGO TERMINA AQUÍ

        # Obtener las derivadas de grads
        dw = grads["dw"]
        db = grads["db"]

        # regla de actualización (≈ 2 líneas de código)
        # w = ...
        # b = ...
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        # TU CÓDIGO TERMINA AQUÍ

        # Guardar los costos
        if i % 100 == 0:
            costs.append(cost)

            # Imprimir el costo cada 100 iteraciones de entrenamiento
            if print_cost:
                print ("Costo después de la iteración %i: %f" %(i, cost))

    params = {"w": w,
              "b": b}

    grads = {"dw": dw,
             "db": db}

    return params, grads, costs

In [ ]:
params, grads, costs = optimize(w, b, X, Y, num_iterations=10, learning_rate=0.009, print_cost=True)

print ("w = " + str(params["w"]))
print ("b = " + str(params["b"]))
print ("dw = " + str(grads["dw"]))
print ("db = " + str(grads["db"]))
print("Costs = " + str(costs))

optimize_test(optimize)

In [ ]:
params, grads, costs = optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False)

print ("w = " + str(params["w"]))
print ("b = " + str(params["b"]))
print ("dw = " + str(grads["dw"]))
print ("db = " + str(grads["db"]))
print("Costs = " + str(costs))

optimize_test(optimize)

<a name='ex-7'></a>
### Ejercicio 7 - predict
La función anterior entrega los parámetros aprendidos w y b. Con ellos podemos predecir las etiquetas de un conjunto de datos X. Implementa la función `predict()`. Hay dos pasos para calcular las predicciones:

1. Calcular $\hat{Y} = A = \sigma(w^T X + b)$

2. Convertir las entradas de A en 0 (si la activación es <= 0.5) o 1 (si la activación es > 0.5), y guardar las predicciones en un vector `Y_prediction`. Si quieres, puedes usar un `if`/`else` dentro de un ciclo `for` (aunque también hay una forma de vectorizarlo).

In [ ]:
# FUNCIÓN CALIFICADA: predict

def predict(w, b, X):
    """
    Predice si la etiqueta es 0 o 1 usando los parámetros aprendidos de la regresión logística (w, b)

    Argumentos:
    w -- pesos, un arreglo de numpy de tamaño (num_px * num_px * 3, 1)
    b -- sesgo, un escalar
    X -- datos de tamaño (num_px * num_px * 3, número de ejemplos)

    Retorna:
    Y_prediction -- un arreglo de numpy (vector) con todas las predicciones (0/1) para los ejemplos de X
    """

    m = X.shape[1]
    Y_prediction = np.zeros((1, m))
    w = w.reshape(X.shape[0], 1)

    # Calcular el vector "A" con las probabilidades de que haya un gato en la imagen
    #(≈ 1 línea de código)
    # A = ...
    # TU CÓDIGO EMPIEZA AQUÍ
    pass  # reemplaza esta línea con tu código
    
    # TU CÓDIGO TERMINA AQUÍ

    for i in range(A.shape[1]):

        # Convertir las probabilidades A[0,i] en predicciones p[0,i]
        #(≈ 4 líneas de código)
        # if A[0, i] > ____ :
        #     Y_prediction[0,i] =
        # else:
        #     Y_prediction[0,i] =
        # TU CÓDIGO EMPIEZA AQUÍ
        pass  # reemplaza esta línea con tu código
        
        # TU CÓDIGO TERMINA AQUÍ

    return Y_prediction

In [ ]:
w = np.array([[0.1124579], [0.23106775]])
b = -0.3
X = np.array([[1., -1.1, -3.2],[1.2, 2., 0.1]])
print ("predicciones = " + str(predict(w, b, X)))

predict_test(predict)

<font color='blue'>

**Lo que debes recordar:**

Implementaste varias funciones que:
- Inicializan (w,b)
- Optimizan la pérdida iterativamente para aprender los parámetros (w,b):
    - Calculando el costo y su gradiente
    - Actualizando los parámetros con descenso por gradiente
- Usan los parámetros aprendidos (w,b) para predecir las etiquetas de un conjunto de ejemplos

<a name='5'></a>
## 5 - Integrar todas las funciones en un modelo ##

Ahora verás cómo se estructura el modelo completo, juntando todos los bloques (las funciones implementadas en las secciones anteriores) en el orden correcto.

<a name='ex-8'></a>
### Ejercicio 8 - model
Implementa la función del modelo. Usa la siguiente notación:
- Y_prediction_test para tus predicciones sobre el conjunto de prueba
- Y_prediction_train para tus predicciones sobre el conjunto de entrenamiento
- parameters, grads, costs para las salidas de optimize()

In [ ]:
# FUNCIÓN CALIFICADA: model

def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5, print_cost=False):
    """
    Construye el modelo de regresión logística llamando las funciones implementadas anteriormente

    Argumentos:
    X_train -- conjunto de entrenamiento, un arreglo de numpy de forma (num_px * num_px * 3, m_train)
    Y_train -- etiquetas de entrenamiento, un arreglo de numpy (vector) de forma (1, m_train)
    X_test -- conjunto de prueba, un arreglo de numpy de forma (num_px * num_px * 3, m_test)
    Y_test -- etiquetas de prueba, un arreglo de numpy (vector) de forma (1, m_test)
    num_iterations -- hiperparámetro: número de iteraciones para optimizar los parámetros
    learning_rate -- hiperparámetro: tasa de aprendizaje usada en la regla de actualización de optimize()
    print_cost -- True para imprimir el costo cada 100 iteraciones

    Retorna:
    d -- diccionario con la información del modelo.
    """
    # (≈ 1 línea de código)
    # inicializar los parámetros en ceros
    # y usar la función "shape" para obtener la primera dimensión de X_train
    # w, b = ...

    #(≈ 1 línea de código)
    # Descenso por gradiente
    # params, grads, costs = ...

    # Obtener los parámetros w y b del diccionario "params"
    # w = ...
    # b = ...

    # Predecir los ejemplos de prueba y de entrenamiento (≈ 2 líneas de código)
    # Y_prediction_test = ...
    # Y_prediction_train = ...

    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    # TU CÓDIGO TERMINA AQUÍ

    # Imprimir la precisión (accuracy) en entrenamiento y prueba
    if print_cost:
        print("Precisión en entrenamiento: {} %".format(100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100))
        print("Precisión en prueba: {} %".format(100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100))


    d = {"costs": costs,
         "Y_prediction_test": Y_prediction_test,
         "Y_prediction_train" : Y_prediction_train,
         "w" : w,
         "b" : b,
         "learning_rate" : learning_rate,
         "num_iterations": num_iterations}

    return d

In [ ]:
model_test(model)

Si pasaste todas las pruebas, ejecuta la siguiente celda para entrenar tu modelo.

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=10000, learning_rate=0.005, print_cost=True)

**Comentario**: la precisión en entrenamiento es cercana al 100%. Esta es una buena verificación: el modelo funciona y tiene suficiente capacidad para ajustarse a los datos de entrenamiento. La precisión en prueba es de 70%. No está mal para un modelo tan sencillo, dado el pequeño conjunto de datos que usamos y que la regresión logística es un clasificador lineal. Más adelante construiremos un clasificador mejor.

También se ve que el modelo claramente está sobreajustando (*overfitting*) los datos de entrenamiento. Más adelante en el curso veremos cómo reducir el sobreajuste, por ejemplo usando regularización. Con el código de abajo (cambiando la variable `index`) puedes ver las predicciones sobre imágenes del conjunto de prueba.

In [ ]:
# Ejemplo de una imagen mal clasificada.
index = 30
plt.imshow(test_set_x[:, index].reshape((num_px, num_px, 3)))
print ("y = " + str(test_set_y[0,index]) + ", el modelo predijo que es una imagen de tipo \"" + classes[int(logistic_regression_model['Y_prediction_test'][0,index])].decode("utf-8") +  "\".")

Grafiquemos también la función de costo a lo largo de las iteraciones.

In [ ]:
# Graficar la curva de aprendizaje (con los costos)
costs = np.squeeze(logistic_regression_model['costs'])
plt.plot(costs)
plt.ylabel('costo')
plt.xlabel('iteraciones (en cientos)')
plt.title("Tasa de aprendizaje = " + str(logistic_regression_model["learning_rate"]))
plt.show()

**Interpretación**:
Se ve cómo el costo disminuye, lo que muestra que los parámetros se están aprendiendo. Sin embargo, se podría entrenar aún más el modelo sobre el conjunto de entrenamiento. Prueba aumentar el número de iteraciones en la celda de arriba y vuelve a ejecutar las celdas. Puede que la precisión en entrenamiento suba pero la precisión en prueba baje. Esto se llama sobreajuste (*overfitting*).

<a name='6'></a>
## 6 - Análisis adicional (opcional) ##

¡Felicitaciones por construir tu primer modelo de clasificación de imágenes! Analicémoslo un poco más y examinemos posibles valores para la tasa de aprendizaje $\alpha$.

#### Elección de la tasa de aprendizaje ####

**Recordatorio**:
Para que el descenso por gradiente funcione debes elegir bien la tasa de aprendizaje. La tasa de aprendizaje $\alpha$ determina qué tan rápido se actualizan los parámetros. Si es demasiado grande, podemos "pasarnos" del valor óptimo. De forma similar, si es demasiado pequeña, necesitaremos demasiadas iteraciones para converger a los mejores valores. Por eso es crucial usar una tasa de aprendizaje bien ajustada.

Comparemos la curva de aprendizaje del modelo con varias tasas de aprendizaje. Ejecuta la celda de abajo; debería tomar alrededor de 1 minuto. Prueba también valores distintos de los tres con los que inicializamos la variable `learning_rates` y observa qué pasa.

In [ ]:
learning_rates = [0.01, 0.001, 0.0001]
models = {}

for lr in learning_rates:
    print ("Entrenando un modelo con tasa de aprendizaje: " + str(lr))
    models[str(lr)] = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=1500, learning_rate=lr, print_cost=False)
    print ('\n' + "-------------------------------------------------------" + '\n')

for lr in learning_rates:
    plt.plot(np.squeeze(models[str(lr)]["costs"]), label=str(models[str(lr)]["learning_rate"]))

plt.ylabel('costo')
plt.xlabel('iteraciones (en cientos)')

legend = plt.legend(loc='upper center', shadow=True)
frame = legend.get_frame()
frame.set_facecolor('0.90')
plt.show()

**Interpretación**:
- Distintas tasas de aprendizaje dan distintos costos y, por lo tanto, distintas predicciones.
- Si la tasa de aprendizaje es demasiado grande (0.01), el costo puede oscilar hacia arriba y hacia abajo. Incluso puede diverger (aunque en este ejemplo, con 0.01 el costo termina llegando a un buen valor).
- Un costo menor no significa un mejor modelo. Hay que revisar si hay sobreajuste, lo cual ocurre cuando la precisión en entrenamiento es mucho mayor que la precisión en prueba.
- En *deep learning* usualmente se recomienda:
    - Elegir la tasa de aprendizaje que mejor minimice la función de costo.
    - Si el modelo sobreajusta, usar otras técnicas para reducir el sobreajuste (las veremos más adelante en el curso).


<a name='7'></a>
## 7 - Prueba con tu propia imagen (opcional) ##

¡Felicitaciones por terminar este notebook! Puedes usar tu propia imagen y ver la predicción del modelo. Para hacerlo:
1. Agrega tu imagen a la carpeta "images" que está en el mismo directorio de este notebook.
2. Cambia el nombre de la imagen en el código de abajo.
3. Ejecuta el código y revisa si el algoritmo acierta (1 = gato, 0 = no-gato).

In [ ]:
# cambia esto por el nombre de tu archivo de imagen
my_image = "ALVARO 2.jpg"

# Preprocesamos la imagen para que sea compatible con el algoritmo.
fname = "images/" + my_image
image = np.array(Image.open(fname).convert("RGB").resize((num_px, num_px)))
plt.imshow(image)
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T
my_predicted_image = predict(logistic_regression_model["w"], logistic_regression_model["b"], image)

print("y = " + str(np.squeeze(my_predicted_image)) + ", el algoritmo predice que es una imagen de tipo \"" + classes[int(np.squeeze(my_predicted_image))].decode("utf-8") +  "\".")

<font color='blue'>

**Lo que debes recordar de este notebook:**
1. El preprocesamiento de los datos es importante.
2. Implementaste cada función por separado: initialize(), propagate(), optimize(). Luego construiste un model().
3. Ajustar la tasa de aprendizaje (que es un ejemplo de "hiperparámetro") puede hacer una gran diferencia en el algoritmo. ¡Verás más ejemplos de esto más adelante en el curso!

Finalmente, te invitamos a probar distintas cosas en este notebook, por ejemplo:
- Cambiar la tasa de aprendizaje y el número de iteraciones
- Probar distintos métodos de inicialización y comparar los resultados
- Probar otros preprocesamientos (centrar los datos, o dividir cada fila por su desviación estándar)

Bibliografía:
- http://www.wildml.com/2015/09/implementing-a-neural-network-from-scratch/
- https://stats.stackexchange.com/questions/211436/why-do-we-normalize-images-by-subtracting-the-datasets-image-mean-and-not-the-c